# Geom-INR-Motion Evaluation Notebook

This notebook provides comprehensive evaluation of the Geom-INR-Motion model, including:

1. **Setup and Model Loading** - Load trained model checkpoints
2. **Metrics Computation** - MPJPE, Smoothness (Jerk), Curvature/Torsion
3. **Visualization** - Trajectory plots, error curves, geometry analysis
4. **Summary Report** - Aggregate metrics and comparisons

## Metrics Overview

- **MPJPE (Mean Per-Joint Position Error)**: $\text{MPJPE} = \frac{1}{TJ}\sum_{t,j}|\hat{\mathbf{p}}_{t,j}-\mathbf{p}_{t,j}|_2$
- **Smoothness (Jerk)**: Average magnitude of third derivative of joint trajectories
- **Curvature**: $\kappa = \frac{|r' \times r''|}{|r'|^3}$ - measures bending of trajectories
- **Torsion**: $\tau = \frac{(r' \times r'') \cdot r'''}{|r' \times r''|^2}$ - measures twisting

## 1. Import Required Libraries

In [ ]:
"""
Import all required libraries for evaluation.
"""
import os
import sys
import json
import numpy as np
from scipy.interpolate import CubicSpline
from typing import Dict, List, Tuple, Optional

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# Visualization
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Add project root to path for imports
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

# Check if local modules are available
try:
    from model import GeomINR, create_model
    from geometry_losses import compute_curvature_torsion, GeometryLoss
    from data_pipeline import create_synthetic_dataset, MotionDataset
    print("✓ Local modules imported successfully")
except ImportError as e:
    print(f"⚠ Warning: Could not import local modules: {e}")
    print("  Some functionality may be limited")

## 2. Configuration and Model Loading

In [ ]:
# ============================================================================
# Configuration
# ============================================================================
CONFIG = {
    'checkpoint_path': 'checkpoints/model_best.pt',
    'num_joints': 24,
    'num_actors': 100,
    'fps': 30.0,
    'test_sequences': 5,
    'test_frames': 100,
}

# ============================================================================
# Load Model
# ============================================================================
def load_model_checkpoint(checkpoint_path: str, device: torch.device) -> nn.Module:
    """
    Load trained GeomINR model from checkpoint.
    
    Args:
        checkpoint_path: Path to .pt checkpoint file
        device: Torch device
    
    Returns:
        Loaded model in eval mode
    """
    try:
        # Create model
        model = create_model(
            variant="base",
            num_joints=CONFIG['num_joints'],
            num_actors=CONFIG['num_actors']
        ).to(device)
        
        # Load checkpoint
        if os.path.exists(checkpoint_path):
            checkpoint = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"✓ Loaded checkpoint from: {checkpoint_path}")
            print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
            print(f"  Best loss: {checkpoint.get('best_loss', 'N/A'):.6f}")
        else:
            print(f"⚠ Checkpoint not found: {checkpoint_path}")
            print("  Using randomly initialized model for demo")
        
        model.eval()
        return model
    
    except Exception as e:
        print(f"Error loading model: {e}")
        print("Creating demo model with random weights...")
        
        # Fallback: create minimal model for demo
        from model import GeomINR
        model = GeomINR(num_joints=CONFIG['num_joints']).to(device)
        model.eval()
        return model

# Try to load model
model = load_model_checkpoint(CONFIG['checkpoint_path'], DEVICE)
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Generate Test Data

Create synthetic test sequences for evaluation (or load real data if available).

In [ ]:
# ============================================================================
# Generate Test Data
# ============================================================================
def create_test_motion(
    num_frames: int = 100,
    num_joints: int = 24,
    motion_type: str = "walking"
) -> np.ndarray:
    """
    Create synthetic motion for testing.
    
    Args:
        num_frames: Number of frames
        num_joints: Number of joints
        motion_type: Type of motion to generate
    
    Returns:
        positions: (T, J, 3) array of joint positions
    """
    T, J = num_frames, num_joints
    t = np.linspace(0, 2 * np.pi, T)
    positions = np.zeros((T, J, 3), dtype=np.float32)
    
    if motion_type == "walking":
        # Simulate walking with cyclic arm/leg motion
        for j in range(J):
            phase = j * np.pi / 6
            positions[:, j, 0] = t / (2 * np.pi) * 2  # Forward
            positions[:, j, 1] = 0.5 + j * 0.05 + 0.05 * np.sin(2 * t + phase)  # Height
            positions[:, j, 2] = 0.03 * np.sin(4 * t + phase)  # Side
    else:
        # Random smooth motion
        np.random.seed(42)
        for j in range(J):
            for d in range(3):
                freqs = np.random.uniform(0.5, 2.0, 3)
                phases = np.random.uniform(0, 2*np.pi, 3)
                for freq, phase in zip(freqs, phases):
                    positions[:, j, d] += 0.1 * np.sin(freq * t + phase)
    
    return positions

# Generate test sequences
print("Generating test sequences...")
test_sequences = []
for i in range(CONFIG['test_sequences']):
    motion_type = "walking" if i % 2 == 0 else "random"
    seq = create_test_motion(
        num_frames=CONFIG['test_frames'],
        num_joints=CONFIG['num_joints'],
        motion_type=motion_type
    )
    test_sequences.append(seq)

print(f"✓ Generated {len(test_sequences)} test sequences")
print(f"  Shape: {test_sequences[0].shape} (T, J, 3)")

# Build interpolators for ground truth queries
def build_interpolator(positions: np.ndarray) -> callable:
    """Build cubic spline interpolator for continuous time queries."""
    T, J, _ = positions.shape
    times = np.arange(T)
    splines = {}
    for j in range(J):
        splines[j] = {}
        for d in range(3):
            splines[j][d] = CubicSpline(times, positions[:, j, d])
    
    def interpolate(t_query):
        t_query = np.atleast_1d(t_query)
        result = np.zeros((len(t_query), J, 3))
        for j in range(J):
            for d in range(3):
                result[:, j, d] = splines[j][d](t_query)
        return result
    
    return interpolate

test_interpolators = [build_interpolator(seq) for seq in test_sequences]
print(f"✓ Built interpolators for {len(test_interpolators)} sequences")

## 4. Model Inference

Query the trained model at continuous time points and compare with ground truth.

In [ ]:
# ============================================================================
# Model Inference
# ============================================================================
def run_model_inference(
    model: torch.nn.Module,
    time_points: np.ndarray,
    actor_id: int = 0,
    device: str = 'cpu'
) -> np.ndarray:
    """
    Run inference on the model at specified time points.
    
    Args:
        model: Trained GeomINR model
        time_points: Array of query times (normalized to [0, 1])
        actor_id: Actor embedding index
        device: Device to run inference on
    
    Returns:
        predictions: (T, J, 3) array of predicted positions
    """
    model.eval()
    with torch.no_grad():
        t_tensor = torch.tensor(time_points, dtype=torch.float32, device=device).unsqueeze(-1)
        actor_ids = torch.full((len(time_points),), actor_id, dtype=torch.long, device=device)
        
        # Model returns (batch, J*3)
        predictions = model(t_tensor, actor_ids)
        predictions = predictions.cpu().numpy()
        
        # Reshape to (T, J, 3)
        T = len(time_points)
        J = predictions.shape[1] // 3
        predictions = predictions.reshape(T, J, 3)
    
    return predictions

# Run inference on all test sequences
print("Running model inference...")
predictions_list = []
ground_truth_list = []

# Use normalized time [0, 1]
time_points = np.linspace(0, 1, CONFIG['test_frames'])

for i, (seq, interpolator) in enumerate(zip(test_sequences, test_interpolators)):
    if model is not None:
        # Model inference
        pred = run_model_inference(model, time_points, actor_id=i, device=device)
    else:
        # Use synthetic prediction for demo (ground truth + noise)
        gt_times = np.linspace(0, CONFIG['test_frames']-1, CONFIG['test_frames'])
        pred = interpolator(gt_times) + np.random.randn(*seq.shape) * 0.02
    
    predictions_list.append(pred)
    ground_truth_list.append(seq)

print(f"✓ Completed inference for {len(predictions_list)} sequences")

## 5. Mean Per-Joint Position Error (MPJPE)

MPJPE is the primary metric for motion prediction. It measures the average Euclidean distance between predicted and ground truth joint positions across all joints and frames.

In [ ]:
# ============================================================================
# MPJPE Computation
# ============================================================================
def compute_mpjpe(pred: np.ndarray, gt: np.ndarray) -> Dict[str, float]:
    """
    Compute Mean Per-Joint Position Error.
    
    Args:
        pred: (T, J, 3) predicted positions
        gt: (T, J, 3) ground truth positions
    
    Returns:
        Dictionary with MPJPE metrics
    """
    assert pred.shape == gt.shape, f"Shape mismatch: {pred.shape} vs {gt.shape}"
    
    # Per-frame, per-joint error
    errors = np.linalg.norm(pred - gt, axis=-1)  # (T, J)
    
    # Overall MPJPE
    mpjpe = np.mean(errors)
    
    # Per-joint MPJPE
    per_joint_mpjpe = np.mean(errors, axis=0)  # (J,)
    
    # Per-frame MPJPE
    per_frame_mpjpe = np.mean(errors, axis=1)  # (T,)
    
    # Compute PA-MPJPE (Procrustes Aligned)
    pa_errors = []
    for t in range(pred.shape[0]):
        # Procrustes alignment per frame
        pred_frame = pred[t]  # (J, 3)
        gt_frame = gt[t]  # (J, 3)
        
        # Center both
        pred_centered = pred_frame - pred_frame.mean(axis=0)
        gt_centered = gt_frame - gt_frame.mean(axis=0)
        
        # SVD for optimal rotation
        H = pred_centered.T @ gt_centered
        U, S, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T
        
        # Handle reflection
        if np.linalg.det(R) < 0:
            Vt[-1, :] *= -1
            R = Vt.T @ U.T
        
        # Apply rotation and compute error
        pred_aligned = pred_centered @ R
        pa_errors.append(np.mean(np.linalg.norm(pred_aligned - gt_centered, axis=-1)))
    
    pa_mpjpe = np.mean(pa_errors)
    
    return {
        'mpjpe': mpjpe,
        'pa_mpjpe': pa_mpjpe,
        'per_joint_mpjpe': per_joint_mpjpe,
        'per_frame_mpjpe': per_frame_mpjpe,
        'std': np.std(errors),
        'max': np.max(errors)
    }

# Compute MPJPE for all sequences
print("Computing MPJPE metrics...")
mpjpe_results = []

for i, (pred, gt) in enumerate(zip(predictions_list, ground_truth_list)):
    result = compute_mpjpe(pred, gt)
    mpjpe_results.append(result)
    print(f"  Sequence {i+1}: MPJPE = {result['mpjpe']*1000:.2f}mm, PA-MPJPE = {result['pa_mpjpe']*1000:.2f}mm")

# Summary statistics
avg_mpjpe = np.mean([r['mpjpe'] for r in mpjpe_results])
avg_pa_mpjpe = np.mean([r['pa_mpjpe'] for r in mpjpe_results])
print(f"\n✓ Average MPJPE: {avg_mpjpe*1000:.2f}mm")
print(f"✓ Average PA-MPJPE: {avg_pa_mpjpe*1000:.2f}mm")

## 6. Smoothness and Jerk Analysis

Analyze motion smoothness using velocity, acceleration, and jerk metrics. Smoother motions have lower jerk values.

In [ ]:
# ============================================================================
# Smoothness Analysis
# ============================================================================
def compute_smoothness_metrics(positions: np.ndarray, dt: float = 1/30) -> Dict[str, float]:
    """
    Compute smoothness metrics including velocity, acceleration, and jerk.
    
    Args:
        positions: (T, J, 3) array of joint positions
        dt: Time step between frames (default: 1/30s for 30fps)
    
    Returns:
        Dictionary with smoothness metrics
    """
    # Compute derivatives using finite differences
    velocity = np.diff(positions, axis=0) / dt  # (T-1, J, 3)
    acceleration = np.diff(velocity, axis=0) / dt  # (T-2, J, 3)
    jerk = np.diff(acceleration, axis=0) / dt  # (T-3, J, 3)
    
    # Compute magnitudes
    vel_mag = np.linalg.norm(velocity, axis=-1)  # (T-1, J)
    acc_mag = np.linalg.norm(acceleration, axis=-1)  # (T-2, J)
    jerk_mag = np.linalg.norm(jerk, axis=-1)  # (T-3, J)
    
    return {
        'mean_velocity': np.mean(vel_mag),
        'max_velocity': np.max(vel_mag),
        'mean_acceleration': np.mean(acc_mag),
        'max_acceleration': np.max(acc_mag),
        'mean_jerk': np.mean(jerk_mag),
        'max_jerk': np.max(jerk_mag),
        'velocity_std': np.std(vel_mag),
        'jerk_per_joint': np.mean(jerk_mag, axis=0)
    }

# Compute smoothness for predictions and ground truth
print("Computing smoothness metrics...")
pred_smoothness = []
gt_smoothness = []

for i, (pred, gt) in enumerate(zip(predictions_list, ground_truth_list)):
    pred_sm = compute_smoothness_metrics(pred)
    gt_sm = compute_smoothness_metrics(gt)
    pred_smoothness.append(pred_sm)
    gt_smoothness.append(gt_sm)
    
    print(f"  Sequence {i+1}:")
    print(f"    Pred Jerk: {pred_sm['mean_jerk']:.4f} | GT Jerk: {gt_sm['mean_jerk']:.4f}")

# Summary comparison
avg_pred_jerk = np.mean([s['mean_jerk'] for s in pred_smoothness])
avg_gt_jerk = np.mean([s['mean_jerk'] for s in gt_smoothness])
jerk_ratio = avg_pred_jerk / (avg_gt_jerk + 1e-8)

print(f"\n✓ Average Prediction Jerk: {avg_pred_jerk:.4f}")
print(f"✓ Average Ground Truth Jerk: {avg_gt_jerk:.4f}")
print(f"✓ Jerk Ratio (Pred/GT): {jerk_ratio:.2f}x")

## 7. Curvature and Torsion Analysis

Compute differential geometry metrics (curvature κ and torsion τ) for each joint trajectory. These are the key regularization losses used in training.

In [ ]:
# ============================================================================
# Curvature and Torsion Computation
# ============================================================================
def compute_curvature_torsion_np(positions: np.ndarray, dt: float = 1/30) -> Dict[str, np.ndarray]:
    """
    Compute curvature and torsion for joint trajectories.
    
    Curvature: κ = |r' × r''| / |r'|³
    Torsion: τ = (r' × r'') · r''' / |r' × r''|²
    
    Args:
        positions: (T, J, 3) array of joint positions
        dt: Time step between frames
    
    Returns:
        Dictionary with curvature and torsion arrays
    """
    T, J, _ = positions.shape
    
    # Compute derivatives (central differences for better accuracy)
    # r' (velocity)
    r_prime = np.zeros_like(positions)
    r_prime[1:-1] = (positions[2:] - positions[:-2]) / (2 * dt)
    r_prime[0] = (positions[1] - positions[0]) / dt
    r_prime[-1] = (positions[-1] - positions[-2]) / dt
    
    # r'' (acceleration)
    r_dprime = np.zeros_like(positions)
    r_dprime[1:-1] = (positions[2:] - 2*positions[1:-1] + positions[:-2]) / (dt**2)
    r_dprime[0] = r_dprime[1]
    r_dprime[-1] = r_dprime[-2]
    
    # r''' (jerk) - using finite differences
    r_tprime = np.zeros_like(positions)
    r_tprime[2:-2] = (positions[4:] - 2*positions[3:-1] + 2*positions[1:-3] - positions[:-4]) / (2 * dt**3)
    r_tprime[:2] = r_tprime[2]
    r_tprime[-2:] = r_tprime[-3]
    
    # Curvature: κ = |r' × r''| / |r'|³
    cross = np.cross(r_prime, r_dprime)  # (T, J, 3)
    cross_norm = np.linalg.norm(cross, axis=-1) + 1e-8  # (T, J)
    r_prime_norm = np.linalg.norm(r_prime, axis=-1) + 1e-8  # (T, J)
    curvature = cross_norm / (r_prime_norm ** 3)  # (T, J)
    
    # Torsion: τ = (r' × r'') · r''' / |r' × r''|²
    # Dot product of cross and r'''
    dot_product = np.sum(cross * r_tprime, axis=-1)  # (T, J)
    torsion = dot_product / (cross_norm ** 2 + 1e-8)  # (T, J)
    
    return {
        'curvature': curvature,
        'torsion': torsion,
        'mean_curvature': np.mean(curvature),
        'max_curvature': np.max(curvature),
        'mean_torsion': np.mean(np.abs(torsion)),
        'max_torsion': np.max(np.abs(torsion))
    }

# Compute curvature/torsion for predictions and ground truth
print("Computing curvature and torsion...")
pred_geometry = []
gt_geometry = []

for i, (pred, gt) in enumerate(zip(predictions_list, ground_truth_list)):
    pred_geom = compute_curvature_torsion_np(pred)
    gt_geom = compute_curvature_torsion_np(gt)
    pred_geometry.append(pred_geom)
    gt_geometry.append(gt_geom)
    
    print(f"  Sequence {i+1}:")
    print(f"    Pred κ: {pred_geom['mean_curvature']:.4f} | GT κ: {gt_geom['mean_curvature']:.4f}")
    print(f"    Pred τ: {pred_geom['mean_torsion']:.4f} | GT τ: {gt_geom['mean_torsion']:.4f}")

# Summary
avg_pred_k = np.mean([g['mean_curvature'] for g in pred_geometry])
avg_gt_k = np.mean([g['mean_curvature'] for g in gt_geometry])
print(f"\n✓ Average Prediction Curvature: {avg_pred_k:.4f}")
print(f"✓ Average Ground Truth Curvature: {avg_gt_k:.4f}")

## 8. Trajectory Visualization

Visualize 3D joint trajectories for selected joints to qualitatively assess prediction quality.

In [ ]:
# ============================================================================
# 3D Trajectory Visualization
# ============================================================================
def plot_3d_trajectory(
    pred: np.ndarray,
    gt: np.ndarray,
    joint_idx: int = 0,
    title: str = "Joint Trajectory"
) -> plt.Figure:
    """
    Plot 3D trajectory for a single joint.
    
    Args:
        pred: (T, J, 3) predicted positions
        gt: (T, J, 3) ground truth positions
        joint_idx: Index of joint to visualize
        title: Plot title
    
    Returns:
        matplotlib Figure
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Extract joint trajectories
    pred_traj = pred[:, joint_idx, :]
    gt_traj = gt[:, joint_idx, :]
    
    # Plot trajectories
    ax.plot(gt_traj[:, 0], gt_traj[:, 1], gt_traj[:, 2], 
            'b-', linewidth=2, label='Ground Truth', alpha=0.7)
    ax.plot(pred_traj[:, 0], pred_traj[:, 1], pred_traj[:, 2], 
            'r--', linewidth=2, label='Prediction', alpha=0.7)
    
    # Mark start and end points
    ax.scatter(*gt_traj[0], c='green', s=100, marker='o', label='Start')
    ax.scatter(*gt_traj[-1], c='red', s=100, marker='x', label='End')
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    ax.legend()
    
    return fig

# Visualize trajectories for selected joints
print("Generating trajectory visualizations...")
seq_idx = 0  # First sequence
joint_names = ['Root', 'Left Hip', 'Right Hip', 'Spine', 'Left Hand', 'Right Hand']
joints_to_vis = [0, 1, 2, 3, 10, 15]  # Representative joints

fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': '3d'})

for idx, (ax, joint_idx) in enumerate(zip(axes.flat, joints_to_vis)):
    if joint_idx >= predictions_list[seq_idx].shape[1]:
        joint_idx = idx  # Fallback if joint doesn't exist
    
    pred_traj = predictions_list[seq_idx][:, joint_idx, :]
    gt_traj = ground_truth_list[seq_idx][:, joint_idx, :]
    
    ax.plot(gt_traj[:, 0], gt_traj[:, 1], gt_traj[:, 2], 'b-', linewidth=2, alpha=0.7)
    ax.plot(pred_traj[:, 0], pred_traj[:, 1], pred_traj[:, 2], 'r--', linewidth=2, alpha=0.7)
    ax.set_title(f'Joint {joint_idx}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

plt.suptitle('Joint Trajectories: Ground Truth (blue) vs Prediction (red)', fontsize=14)
plt.tight_layout()
plt.show()

print("✓ Trajectory visualization complete")

## 9. MPJPE Over Prediction Horizon

Analyze how prediction error grows over time - a key indicator of model performance for long-horizon prediction.

In [ ]:
# ============================================================================
# MPJPE Over Prediction Horizon
# ============================================================================
def compute_horizon_mpjpe(
    predictions: List[np.ndarray],
    ground_truths: List[np.ndarray]
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute MPJPE at each time step across all sequences.
    
    Args:
        predictions: List of (T, J, 3) arrays
        ground_truths: List of (T, J, 3) arrays
    
    Returns:
        time_steps: Array of time step indices
        mpjpe_per_frame: Mean MPJPE at each time step
    """
    T = predictions[0].shape[0]
    mpjpe_per_frame = np.zeros(T)
    
    for pred, gt in zip(predictions, ground_truths):
        errors = np.linalg.norm(pred - gt, axis=-1)  # (T, J)
        frame_mpjpe = np.mean(errors, axis=1)  # (T,)
        mpjpe_per_frame += frame_mpjpe
    
    mpjpe_per_frame /= len(predictions)
    time_steps = np.arange(T)
    
    return time_steps, mpjpe_per_frame

# Compute horizon MPJPE
time_steps, mpjpe_horizon = compute_horizon_mpjpe(predictions_list, ground_truth_list)

# Plot MPJPE over horizon
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(time_steps, mpjpe_horizon * 1000, 'b-', linewidth=2, label='MPJPE')
ax.fill_between(time_steps, 0, mpjpe_horizon * 1000, alpha=0.3)

# Add reference lines for common prediction horizons
for t, label in [(25, '80ms'), (50, '160ms'), (75, '240ms')]:
    if t < len(time_steps):
        ax.axvline(x=t, color='gray', linestyle='--', alpha=0.5)
        ax.text(t+1, ax.get_ylim()[1]*0.9, label, fontsize=10)

ax.set_xlabel('Frame', fontsize=12)
ax.set_ylabel('MPJPE (mm)', fontsize=12)
ax.set_title('MPJPE Over Prediction Horizon', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Report key horizons
print("MPJPE at key prediction horizons:")
for t in [10, 25, 50, 75, 100]:
    if t < len(mpjpe_horizon):
        print(f"  Frame {t:3d} ({t*33.3:.0f}ms): {mpjpe_horizon[t]*1000:.2f}mm")

## 10. Per-Joint Error Analysis

Analyze which joints have the highest prediction error to identify model weaknesses.

In [ ]:
# ============================================================================
# Per-Joint Error Analysis
# ============================================================================
# Aggregate per-joint MPJPE across all sequences
num_joints = predictions_list[0].shape[1]
per_joint_errors = np.zeros(num_joints)

for result in mpjpe_results:
    per_joint_errors += result['per_joint_mpjpe']
per_joint_errors /= len(mpjpe_results)

# Sort joints by error
joint_order = np.argsort(per_joint_errors)[::-1]

# Plot per-joint MPJPE
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart (sorted by error)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, num_joints))
bars = ax1.bar(range(num_joints), per_joint_errors[joint_order] * 1000, color=colors)
ax1.set_xlabel('Joint (sorted by error)', fontsize=12)
ax1.set_ylabel('MPJPE (mm)', fontsize=12)
ax1.set_title('Per-Joint MPJPE (Sorted)', fontsize=14)
ax1.set_xticks(range(0, num_joints, max(1, num_joints//10)))

# Heatmap visualization
joint_grid = per_joint_errors.reshape(-1, 1)
im = ax2.imshow(joint_grid * 1000, cmap='hot', aspect='auto')
ax2.set_xlabel('', fontsize=12)
ax2.set_ylabel('Joint Index', fontsize=12)
ax2.set_title('Joint Error Heatmap', fontsize=14)
plt.colorbar(im, ax=ax2, label='MPJPE (mm)')

plt.tight_layout()
plt.show()

# Report top 5 worst joints
print("Top 5 joints with highest error:")
for i, j in enumerate(joint_order[:5]):
    print(f"  {i+1}. Joint {j}: {per_joint_errors[j]*1000:.2f}mm")

## 11. Curvature Distribution Comparison

Compare the distribution of curvature values between predictions and ground truth to assess how well the model captures motion smoothness.

In [ ]:
# ============================================================================
# Curvature Distribution Comparison
# ============================================================================
# Collect curvature values
pred_curvatures = np.concatenate([g['curvature'].flatten() for g in pred_geometry])
gt_curvatures = np.concatenate([g['curvature'].flatten() for g in gt_geometry])

# Remove outliers for visualization (clip at 99th percentile)
clip_val = np.percentile(np.concatenate([pred_curvatures, gt_curvatures]), 99)
pred_curvatures_clipped = np.clip(pred_curvatures, 0, clip_val)
gt_curvatures_clipped = np.clip(gt_curvatures, 0, clip_val)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram comparison
bins = np.linspace(0, clip_val, 50)
ax1.hist(gt_curvatures_clipped, bins=bins, alpha=0.5, label='Ground Truth', density=True)
ax1.hist(pred_curvatures_clipped, bins=bins, alpha=0.5, label='Prediction', density=True)
ax1.set_xlabel('Curvature (κ)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Curvature Distribution', fontsize=14)
ax1.legend()

# Q-Q plot
percentiles = np.linspace(0, 100, 101)
gt_quantiles = np.percentile(gt_curvatures, percentiles)
pred_quantiles = np.percentile(pred_curvatures, percentiles)

ax2.scatter(gt_quantiles, pred_quantiles, alpha=0.5, s=20)
max_val = max(gt_quantiles.max(), pred_quantiles.max())
ax2.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Match')
ax2.set_xlabel('Ground Truth Curvature Quantiles', fontsize=12)
ax2.set_ylabel('Prediction Curvature Quantiles', fontsize=12)
ax2.set_title('Curvature Q-Q Plot', fontsize=14)
ax2.legend()

plt.tight_layout()
plt.show()

# Compute distribution statistics
print("Curvature Statistics:")
print(f"  Ground Truth - Mean: {np.mean(gt_curvatures):.4f}, Std: {np.std(gt_curvatures):.4f}")
print(f"  Prediction   - Mean: {np.mean(pred_curvatures):.4f}, Std: {np.std(pred_curvatures):.4f}")

## 12. Skeleton Visualization

Visualize skeleton poses at different time steps to qualitatively compare prediction and ground truth.

In [ ]:
# ============================================================================
# Skeleton Visualization
# ============================================================================
# Define skeleton connections (SMPL-like topology)
SKELETON_CONNECTIONS = [
    (0, 1), (0, 2), (0, 3),  # Root to hips and spine
    (1, 4), (2, 5), (3, 6),  # Hips to knees, spine to chest
    (4, 7), (5, 8), (6, 9),  # Knees to ankles, chest to neck
    (9, 12), (9, 13), (9, 14),  # Neck to head and shoulders
    (12, 15), (13, 16), (14, 17),  # Shoulders to elbows
    (15, 18), (16, 19), (17, 20),  # Elbows to wrists
    (7, 10), (8, 11),  # Ankles to feet
]

def plot_skeleton_3d(
    ax: plt.Axes,
    positions: np.ndarray,
    connections: List[Tuple[int, int]],
    color: str = 'blue',
    alpha: float = 1.0,
    label: str = None
):
    """Plot a 3D skeleton on given axes."""
    num_joints = positions.shape[0]
    
    # Plot joints
    ax.scatter(positions[:, 0], positions[:, 1], positions[:, 2], 
               c=color, s=50, alpha=alpha, label=label)
    
    # Plot bones
    for i, j in connections:
        if i < num_joints and j < num_joints:
            ax.plot([positions[i, 0], positions[j, 0]],
                    [positions[i, 1], positions[j, 1]],
                    [positions[i, 2], positions[j, 2]],
                    c=color, linewidth=2, alpha=alpha)

# Visualize skeleton at key frames
seq_idx = 0
frames_to_show = [0, 25, 50, 75, 99]
frames_to_show = [f for f in frames_to_show if f < predictions_list[seq_idx].shape[0]]

fig = plt.figure(figsize=(16, 4))

for idx, frame in enumerate(frames_to_show):
    ax = fig.add_subplot(1, len(frames_to_show), idx + 1, projection='3d')
    
    gt_pose = ground_truth_list[seq_idx][frame]
    pred_pose = predictions_list[seq_idx][frame]
    
    # Use simplified connections based on actual joint count
    num_joints = gt_pose.shape[0]
    simple_connections = [(i, i+1) for i in range(min(num_joints-1, 10))]
    
    plot_skeleton_3d(ax, gt_pose, simple_connections, color='blue', alpha=0.6, label='GT')
    plot_skeleton_3d(ax, pred_pose, simple_connections, color='red', alpha=0.6, label='Pred')
    
    ax.set_title(f'Frame {frame}')
    ax.set_xlim(gt_pose[:, 0].min() - 0.5, gt_pose[:, 0].max() + 0.5)
    ax.set_ylim(gt_pose[:, 1].min() - 0.5, gt_pose[:, 1].max() + 0.5)
    ax.set_zlim(gt_pose[:, 2].min() - 0.5, gt_pose[:, 2].max() + 0.5)
    
    if idx == 0:
        ax.legend()

plt.suptitle('Skeleton Comparison: Ground Truth (blue) vs Prediction (red)', fontsize=14)
plt.tight_layout()
plt.show()

print("✓ Skeleton visualization complete")

## 13. Metrics Summary Table

Compile all metrics into a comprehensive summary table for easy comparison.

In [ ]:
# ============================================================================
# Metrics Summary Table
# ============================================================================
# Compile all metrics
summary_data = {
    'Metric': [
        'MPJPE (mm)',
        'PA-MPJPE (mm)',
        'Mean Jerk (Pred)',
        'Mean Jerk (GT)',
        'Jerk Ratio',
        'Mean Curvature (Pred)',
        'Mean Curvature (GT)',
        'Mean Torsion (Pred)',
        'Mean Torsion (GT)',
    ],
    'Value': [
        f"{avg_mpjpe * 1000:.2f}",
        f"{avg_pa_mpjpe * 1000:.2f}",
        f"{avg_pred_jerk:.4f}",
        f"{avg_gt_jerk:.4f}",
        f"{jerk_ratio:.2f}x",
        f"{avg_pred_k:.4f}",
        f"{avg_gt_k:.4f}",
        f"{np.mean([g['mean_torsion'] for g in pred_geometry]):.4f}",
        f"{np.mean([g['mean_torsion'] for g in gt_geometry]):.4f}",
    ]
}

# Create and display table
print("=" * 50)
print("EVALUATION SUMMARY")
print("=" * 50)
print(f"{'Metric':<30} {'Value':>15}")
print("-" * 50)
for metric, value in zip(summary_data['Metric'], summary_data['Value']):
    print(f"{metric:<30} {value:>15}")
print("=" * 50)

# Per-sequence breakdown
print("\nPer-Sequence Breakdown:")
print("-" * 60)
print(f"{'Seq':<5} {'MPJPE (mm)':<12} {'PA-MPJPE (mm)':<14} {'Jerk':<10} {'Curvature':<12}")
print("-" * 60)
for i in range(len(predictions_list)):
    print(f"{i+1:<5} {mpjpe_results[i]['mpjpe']*1000:<12.2f} "
          f"{mpjpe_results[i]['pa_mpjpe']*1000:<14.2f} "
          f"{pred_smoothness[i]['mean_jerk']:<10.4f} "
          f"{pred_geometry[i]['mean_curvature']:<12.4f}")

## 14. Export Results

Export predictions and metrics to various formats for further analysis or visualization in external tools.

In [ ]:
# ============================================================================
# Export Results
# ============================================================================
# Create output directory
output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

# Export metrics to JSON
metrics_output = {
    'summary': {
        'avg_mpjpe_mm': float(avg_mpjpe * 1000),
        'avg_pa_mpjpe_mm': float(avg_pa_mpjpe * 1000),
        'avg_jerk_pred': float(avg_pred_jerk),
        'avg_jerk_gt': float(avg_gt_jerk),
        'avg_curvature_pred': float(avg_pred_k),
        'avg_curvature_gt': float(avg_gt_k),
    },
    'per_sequence': [
        {
            'sequence_id': i,
            'mpjpe_mm': float(mpjpe_results[i]['mpjpe'] * 1000),
            'pa_mpjpe_mm': float(mpjpe_results[i]['pa_mpjpe'] * 1000),
            'mean_jerk': float(pred_smoothness[i]['mean_jerk']),
            'mean_curvature': float(pred_geometry[i]['mean_curvature']),
        }
        for i in range(len(predictions_list))
    ],
    'config': CONFIG
}

metrics_path = output_dir / 'evaluation_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_output, f, indent=2)
print(f"✓ Exported metrics to {metrics_path}")

# Export predictions to NPZ
for i, (pred, gt) in enumerate(zip(predictions_list, ground_truth_list)):
    npz_path = output_dir / f'sequence_{i:03d}.npz'
    np.savez_compressed(
        npz_path,
        predictions=pred,
        ground_truth=gt,
        mpjpe=mpjpe_results[i]['mpjpe'],
        per_frame_mpjpe=mpjpe_results[i]['per_frame_mpjpe']
    )

print(f"✓ Exported {len(predictions_list)} sequences to {output_dir}")

# Save summary plot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# MPJPE histogram
axes[0, 0].bar(range(len(mpjpe_results)), 
               [r['mpjpe']*1000 for r in mpjpe_results])
axes[0, 0].set_xlabel('Sequence')
axes[0, 0].set_ylabel('MPJPE (mm)')
axes[0, 0].set_title('MPJPE per Sequence')

# MPJPE over horizon
axes[0, 1].plot(time_steps, mpjpe_horizon * 1000)
axes[0, 1].set_xlabel('Frame')
axes[0, 1].set_ylabel('MPJPE (mm)')
axes[0, 1].set_title('MPJPE over Horizon')

# Per-joint MPJPE
axes[1, 0].bar(range(num_joints), per_joint_errors * 1000)
axes[1, 0].set_xlabel('Joint')
axes[1, 0].set_ylabel('MPJPE (mm)')
axes[1, 0].set_title('Per-Joint MPJPE')

# Curvature comparison
axes[1, 1].hist(gt_curvatures_clipped, bins=30, alpha=0.5, label='GT')
axes[1, 1].hist(pred_curvatures_clipped, bins=30, alpha=0.5, label='Pred')
axes[1, 1].set_xlabel('Curvature')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Curvature Distribution')
axes[1, 1].legend()

plt.tight_layout()
fig.savefig(output_dir / 'evaluation_summary.png', dpi=150)
print(f"✓ Saved summary plot to {output_dir / 'evaluation_summary.png'}")

## 15. Conclusion

This notebook provides a comprehensive evaluation of the Geom-INR-Motion model including:

1. **MPJPE Analysis**: Standard motion prediction accuracy metrics
2. **Smoothness Metrics**: Velocity, acceleration, and jerk analysis
3. **Geometry Losses**: Curvature and torsion that drive the model's regularization
4. **Visualizations**: 3D trajectories and skeleton comparisons
5. **Export**: Results saved for further analysis

### Key Findings

- The model's prediction accuracy (MPJPE) is evaluated across all test sequences
- Smoothness is compared between predictions and ground truth
- The curvature/torsion regularization effect can be assessed via distribution comparisons

### Next Steps

1. Train on full AMASS dataset for production-quality results
2. Tune geometry loss weights based on evaluation metrics
3. Experiment with longer prediction horizons
4. Compare against baseline methods (RNNs, Transformers)